# Composição de modelos

Composição de modelos é, basicamente, definir modelos Pydantic em que um ou mais campos são, eles mesmos, modelos Pydantic.

Não é diferente da composição comum (classes Pydantic são classes Python) — a diferença principal é que o Pydantic sabe serializar e desserializar esses campos, já que são modelos Pydantic.

Vamos a alguns exemplos.

Digamos que queremos uma classe `Circle` com um campo `center` e um campo `radius`. Antes, definíamos `center` só como uma tupla.

Agora queremos um círculo 2D, nomeando explicitamente as coordenadas da origem como `x` e `y`:

In [1]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError


class Point2D(BaseModel):
    x: float = 0
    y: float = Field(default=1, gt=0)

class Circle2D(BaseModel):
    center: Point2D
    radius: float = Field(default=1, gt=0)

Para criar uma instância de `Circle` pela abordagem baseada em classes, precisamos especificar o centro usando explicitamente o modelo `Point2D`:

In [2]:
c = Circle2D(center=Point2D(x=1, y=1), radius=2)

Vejamos a representação:

In [3]:
print(f"c: {c!r}")

c: Circle2D(center=Point2D(x=1.0, y=1.0), radius=2.0)


O Pydantic mostra o modelo aninhado, e o mesmo acontece na serialização: ela gera estruturas de dados aninhadas.

Serializando para objetos Python:

In [4]:
print(f"c.model_dump(): {c.model_dump()!r}")

c.model_dump(): {'center': {'x': 1.0, 'y': 1.0}, 'radius': 2.0}


Repare que `Point2D` foi serializado como dicionário: o resultado inteiro é um dicionário em que `center` é outro dicionário.

O mesmo vale para a serialização em JSON:

In [5]:
print(f"c.model_dump_json(): {c.model_dump_json()!r}")

c.model_dump_json(): '{"center":{"x":1.0,"y":1.0},"radius":2.0}'


Isso já sugere que a desserialização a partir de dicionários ou JSON funciona do mesmo jeito:

> 📝 **Nota de revisão:** com o `Point2D` como definido acima (`y` com `gt=0`), os dados a seguir (`"y": -5`) **falham** na validação — o próprio notebook original registra esse `ValidationError`. O `gt=0` em `y` parece ter sido um descuido (uma coordenada pode ser negativa; quem precisa ser positivo é o raio). Mostramos o erro e, em seguida, corrigimos `Point2D` para o resto da aula.

In [6]:
data = {
    "center": {
        "x": 5, 
        "y": -5
    },
    "radius": 0.00001
}

try:
    Circle2D.model_validate(data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Circle2D
center.y
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


Com `y` sem a restrição, o mesmo dado valida:

In [7]:
class Point2D(BaseModel):
    x: float = 0
    y: float = 0            # corrigido: sem gt=0

class Circle2D(BaseModel):
    center: Point2D
    radius: float = Field(default=1, gt=0)

c = Circle2D.model_validate(data)
print(f"c: {c!r}")

c: Circle2D(center=Point2D(x=5.0, y=-5.0), radius=1e-05)


E, de forma análoga, com dados JSON:

In [8]:
data = """
{
    "center": {
        "x": 5, 
        "y": -5
    },
    "radius": 10
}
"""

c = Circle2D.model_validate_json(data)
print(f"c: {c!r}")

c: Circle2D(center=Point2D(x=5.0, y=-5.0), radius=10.0)


E, claro, acessamos os campos com notação de ponto, como em classes comuns:

In [9]:
print(f"c.center: {c.center!r}")

c.center: Point2D(x=5.0, y=-5.0)


In [10]:
print(f"c.center.x: {c.center.x!r}")

c.center.x: 5.0


Uma vantagem da composição é que cada modelo usado pode ser configurado de forma independente.

Por exemplo, suponha que recebemos este JSON de uma chamada a alguma API.

Só nos interessa uma parte da informação, então podemos criar modelos que ignoram os dados extras.

In [11]:
json_data = """
{
    "firstName": "David",
    "lastName": "Hilbert",
    "contactInfo": {
        "email": "d.hilbert@spectral-theory.com",
        "homePhone": {
            "countryCode": 49,
            "areaCode": 551,
            "localPhoneNumber": 123456789
        }
    },
    "personalInfo": {
        "nationality": "German",
        "born": {
            "date": "1862-01-23",
            "place": {
                "city": "Konigsberg",
                "country": "Prussia"
            }
        },
        "died": {
            "date": "1943-02-14",
            "place": {
                "city": "Gottingen",
                "country": "Germany"
            }
        }
    },
    "awards": ["Lobachevsky Prize", "Bolyai Prize", "ForMemRS"],
    "notableStudents": ["von Neumann", "Weyl", "Courant", "Zermelo"]
}
"""

Suponha que só nos interessam o nome, o sobrenome, o e-mail, os dados de nascimento, os prêmios e os alunos notáveis.

Para modelar isso com Pydantic, decompomos a estrutura:

> 📝 **Nota de revisão:** o texto cita *prêmios* (`awards`), mas o modelo abaixo não tem esse campo — ele é ignorado como os demais extras. Também vale saber que `extra="ignore"` já é o **padrão** do Pydantic; deixá-lo explícito só documenta a intenção. (`populate_by_name` continua funcionando; nas versões recentes os nomes preferidos são `validate_by_name`/`validate_by_alias` — ver seção 4.)

In [12]:
from pydantic import EmailStr, PastDate
from pydantic.alias_generators import to_camel


class ContactInfo(BaseModel):
    model_config = ConfigDict(extra="ignore")

    email: EmailStr | None = None

class PlaceInfo(BaseModel):
    city: str
    country: str
    
class PlaceDateInfo(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    
    date_: PastDate = Field(alias="date")
    place: PlaceInfo
    
class PersonalInfo(BaseModel):
    model_config = ConfigDict(extra="ignore")

    nationality: str
    born: PlaceDateInfo

class Person(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel, populate_by_name=True, extra="ignore")
    
    first_name: str
    last_name: str
    contact_info: ContactInfo
    personal_info: PersonalInfo
    notable_students: list[str] = []

In [13]:
p = Person.model_validate_json(json_data)
print(f"p: {p!r}")

p: Person(first_name='David', last_name='Hilbert', contact_info=ContactInfo(email='d.hilbert@spectral-theory.com'), personal_info=PersonalInfo(nationality='German', born=PlaceDateInfo(date_=datetime.date(1862, 1, 23), place=PlaceInfo(city='Konigsberg', country='Prussia'))), notable_students=['von Neumann', 'Weyl', 'Courant', 'Zermelo'])


Ao serializar o modelo, tudo fica aninhado (como na entrada) e os campos extras que a aplicação não precisava foram descartados:

In [14]:
print(f"p.model_dump_json(by_alias=True, indent=2):\n{p.model_dump_json(by_alias=True, indent=2)}")

p.model_dump_json(by_alias=True, indent=2):
{
  "firstName": "David",
  "lastName": "Hilbert",
  "contactInfo": {
    "email": "d.hilbert@spectral-theory.com"
  },
  "personalInfo": {
    "nationality": "German",
    "born": {
      "date": "1862-01-23",
      "place": {
        "city": "Konigsberg",
        "country": "Prussia"
      }
    }
  },
  "notableStudents": [
    "von Neumann",
    "Weyl",
    "Courant",
    "Zermelo"
  ]
}


Como exercício, vamos garantir que a lista `notableStudents` esteja sempre em ordem alfabética.

Fazemos isso com um validador customizado — e uso um tipo `Annotated`, porque posso querer reaproveitar uma lista de strings ordenada em outros pontos da aplicação.

In [15]:
from typing import Annotated
from pydantic import AfterValidator


# using an AfterValidator, so guaranteed value will be a list (empty or strings)
SortedStringList = Annotated[list[str], AfterValidator(lambda value: sorted(value, key=str.casefold))]

class Person(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel, populate_by_name=True, extra="ignore")
    
    first_name: str
    last_name: str
    contact_info: ContactInfo
    personal_info: PersonalInfo
    notable_students: SortedStringList = []

In [16]:
p = Person.model_validate_json(json_data)
print(f"p: {p!r}")

p: Person(first_name='David', last_name='Hilbert', contact_info=ContactInfo(email='d.hilbert@spectral-theory.com'), personal_info=PersonalInfo(nationality='German', born=PlaceDateInfo(date_=datetime.date(1862, 1, 23), place=PlaceInfo(city='Konigsberg', country='Prussia'))), notable_students=['Courant', 'von Neumann', 'Weyl', 'Zermelo'])


In [17]:
print(f"p.model_dump_json(by_alias=True, indent=2):\n{p.model_dump_json(by_alias=True, indent=2)}")

p.model_dump_json(by_alias=True, indent=2):
{
  "firstName": "David",
  "lastName": "Hilbert",
  "contactInfo": {
    "email": "d.hilbert@spectral-theory.com"
  },
  "personalInfo": {
    "nationality": "German",
    "born": {
      "date": "1862-01-23",
      "place": {
        "city": "Konigsberg",
        "country": "Prussia"
      }
    }
  },
  "notableStudents": [
    "Courant",
    "von Neumann",
    "Weyl",
    "Zermelo"
  ]
}


Este é um "truque" que uso com frequência quando uma consulta a uma API devolve muito mais informação do que preciso: elimino todos os dados extras com `extra="ignore"` em qualquer um dos modelos usados pelo modelo principal.